# 🌲 Module 14: Fire, Logging & Road Encroachment Pattern Analysis

Welcome to **Module 14** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Landscape Ecology Patterns**: Understand how roads, fires, logging, and clearcuts generate distinct spatial geometries.
2. **Morphological Extraction**: Compute patch metrics: Linearity, Circularity, Solidity, and Fractal Dimension.
3. **Disturbance Driver Attribution**: Automatically classify each deforestation polygon into its anthropological or natural driver.
4. **Edge Effect & Fragmentation**: Model the 100m degraded edge effect buffer and quantify intact core forest losses.

In [ ]:
import os
import sys
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import cv2

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_13_severity_classification.severity_indices import DisturbanceSeverityCalculator
from modules.module_14_pattern_analysis.morphological_analyzer import (
    PatchMorphologyExtractor, ForestFragmentationAnalyzer
)
from modules.module_14_pattern_analysis.driver_classifier import DisturbanceDriverClassifier

%matplotlib inline
print("✅ Module 14 Pattern Analysis Environment Initialized!")

## 1. Extract Patch Morphology from Test Imagery

We extract contours and compute geometric metrics for every connected disturbance polygon.

In [ ]:
calc = DisturbanceSeverityCalculator()
extractor = PatchMorphologyExtractor(min_patch_pixels=4, pixel_res_meters=30.0)
driver_clf = DisturbanceDriverClassifier()

test_b = sorted(glob.glob("../dataset/test/before/*.tif"))
test_a = sorted(glob.glob("../dataset/test/after/*.tif"))
test_m = sorted(glob.glob("../dataset/test/masks/*.tif"))

with rasterio.open(test_b[0]) as s1:
    t1 = s1.read()
with rasterio.open(test_a[0]) as s2:
    t2 = s2.read()
with rasterio.open(test_m[0]) as sm:
    mask = sm.read(1)

idx = calc.compute_all_indices(t1, t2)
patches, labeled_mask = extractor.extract_patch_metrics(mask, dnbr_raster=idx["dnbr"])
print(f"Extracted {len(patches)} disturbance patches from Scene 1")

## 2. Disturbance Driver Attribution

Classify each patch into: Road Encroachment, Wildfire Scar, Selective Logging, or Agricultural Clearcut.

In [ ]:
df_patches = driver_clf.classify_all_patches(patches)
driver_stats = driver_clf.compute_driver_statistics(df_patches)

for driver, s in driver_stats.items():
    print(f"{driver:<24}: {s['patch_count']:>3} patches | {s['total_area_ha']:>7.2f} ha ({s['area_percentage']:>5.2f}%)")

## 3. Forest Fragmentation & 100m Edge Buffer Analysis

Quantifies Core Forest (>100m from edge) vs. Degraded Edge Forest (<=100m from edge).

In [ ]:
frag_analyzer = ForestFragmentationAnalyzer(edge_buffer_meters=100.0, pixel_res_meters=30.0)
forest_t1 = (calc.compute_ndvi(t1[3], t1[2]) >= 0.50).astype(np.uint8)

frag_metrics, frag_map = frag_analyzer.analyze_fragmentation(forest_t1, mask)
print(f"Total Remaining Forest : {frag_metrics['total_forest_remaining_ha']:.1f} ha")
print(f"Core Forest (>100m)    : {frag_metrics['core_forest_ha']:.1f} ha ({frag_metrics['core_forest_percentage']:.1f}%)")
print(f"Edge Forest (<=100m)   : {frag_metrics['edge_forest_ha']:.1f} ha ({frag_metrics['edge_forest_percentage']:.1f}%)")